### This demo showcases the implementation of story RSPY-808 (Implement STAC view of EDRS sessions)
See https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-808


In [1]:
# Init environment before running a demo notebook.
from resources.utils import *  
from resources.dask_utils import *
import pprint
pp = pprint.PrettyPrinter(indent=2, width=80, sort_dicts=False, compact=True)

auxip_client, cadip_client, catalog_client, staging_client, prip_client, edrs_client = init_demo()

Auxip service: http://rs-server-adgs:8000/auxip
PRIP service: http://rs-server-prip:8000/prip
CADIP service: http://rs-server-cadip:8000/cadip
EDRS service: http://rs-server-edrs:8000/edrs
Catalog service: http://rs-server-catalog:8000
Staging service: http://rs-server-staging:8000
DPR service: http://rs-dpr-service:8000


In [2]:
# STAC API landing page. /edrs/
edrs_client.get_landing()

{'type': 'Catalog',
 'id': 'stac-fastapi',
 'stac_version': '1.1.0',
 'description': 'Edrs collections of Copernicus Reference System Python',
 'links': [{'rel': 'self',
   'href': 'http://rs-server-edrs:8000/edrs/',
   'type': 'application/json'},
  {'rel': 'root',
   'href': 'http://rs-server-edrs:8000/edrs/',
   'type': 'application/json',
   'title': 'RS-PYTHON Edrs collections'},
  {'rel': 'data',
   'href': 'http://rs-server-edrs:8000/edrs/collections',
   'type': 'application/json',
   'title': 'Collections available for this Catalog'},
  {'rel': 'conformance',
   'href': 'http://rs-server-edrs:8000/edrs/conformance',
   'type': 'application/json',
   'title': 'STAC/OGC conformance classes implemented by this server'},
  {'rel': 'search',
   'href': 'http://rs-server-edrs:8000/edrs/search',
   'type': 'application/geo+json',
   'title': 'STAC search [GET]',
   'method': 'GET'},
  {'rel': 'search',
   'href': 'http://rs-server-edrs:8000/edrs/search',
   'type': 'application/geo+j

In [3]:
# STAC collections the user has permission to access. '/edrs/collections'
collections = edrs_client.get_collections()
for c in collections:
    pp.pprint(c.to_dict()['id'])

's1_pedc'
's2_pedc'


In [4]:
# Queryable fields. '/edrs/queryables'
general_queryables = edrs_client.get_queryables()
assert isinstance(general_queryables, dict)
pprint.pp(general_queryables)

{'$id': 'http://rs-server-edrs:8000/edrs/queryables',
 'type': 'object',
 'title': 'STAC Queryables.',
 '$schema': 'http://json-schema.org/draft-07/schema#',
 'properties': {'id': {'type': 'string',
                       'title': 'id',
                       'format': 'string',
                       'pattern': None,
                       'description': 'STAC Item id (session identifier)',
                       'enum': None},
                'collection': {'type': 'string',
                               'title': 'collection',
                               'format': 'string',
                               'pattern': None,
                               'description': 'Collection id',
                               'enum': None},
                'datetime': {'type': 'string',
                             'title': 'datetime',
                             'format': 'date-time',
                             'pattern': None,
                             'description': 'Nominal datetime

In [5]:
fields = list(general_queryables["properties"].keys())
print(fields)

['id', 'collection', 'datetime', 'start_datetime', 'end_datetime', 'published', 'platform', 'constellation']


In [6]:
collection_queryables = edrs_client.get_collection_queryables(collection_id='s1_pedc')
assert isinstance(collection_queryables, dict)
fields = list(collection_queryables["properties"].keys())
print(fields)

['id', 'collection', 'datetime', 'start_datetime', 'end_datetime', 'published', 'platform', 'constellation']


In [7]:
# STAC collections the user has permission to access. '/edrs/collections/{collectionId}'
edrs_client.get_collection(collection_id='s1_pedc')

<CollectionClient id=s1_pedc>

In [8]:
# STAC collections the user has permission to access. '/edrs/collections/{collectionId}'
edrs_client.get_collection(collection_id='s2_pedc')

<CollectionClient id=s2_pedc>

In [9]:
#'/edrs/collections/{collectionId}/items'
items_list = list(edrs_client.get_items(collection_id="s1_pedc"))
pp.pprint(items_list)

14:33:54.386 [INFO] (rs_client.rs_client) Retrieving all items from collection 's1_pedc'.


[ <Item id=DCS_02_202502131123000000987654>,
  <Item id=DCS_01_202501270945000000112233>]


In [10]:
#'/edrs/collections/{collectionId}/items/{featureId}'
edrs_client.get_item(collection_id='s1_pedc', item_id="DCS_02_202502131123000000987654")

<Item id=DCS_02_202502131123000000987654>

In [11]:
#'/edrs/collections/{collectionId}/items/{featureId}'
edrs_client.get_item(collection_id='s1_pedc', item_id="DCS_01_202501270945000000112233")

<Item id=DCS_01_202501270945000000112233>

In [12]:
items_iter = edrs_client.get_items(collection_id="s1_pedc", sortby='+published', limit=1, page=1)
items_list = list(items_iter)
assert len(items_list) == 1
pp.pprint(items_list)
items_iter = edrs_client.get_items(collection_id="s1_pedc", sortby='+published', limit=1, page=2)
items_list = list(items_iter)
assert len(items_list) == 1
pp.pprint(items_list)

14:33:54.800 [INFO] (rs_client.rs_client) Retrieving items from collection 's1_pedc' with query params: {'sortby': '+published', 'limit': 1, 'page': 1}.
14:33:54.925 [INFO] (rs_client.rs_client) Retrieving items from collection 's1_pedc' with query params: {'sortby': '+published', 'limit': 1, 'page': 2}.


[<Item id=DCS_01_202501270945000000112233>]
[<Item id=DCS_02_202502131123000000987654>]


In [13]:
items_iter = edrs_client.get_items(collection_id="s1_pedc", sortby='+published', limit=2, page=1)
items_list = list(items_iter)
assert len(items_list) == 2
pp.pprint(items_list)
items_iter = edrs_client.get_items(collection_id="s1_pedc", sortby='+published', limit=2, page=2)
items_list = list(items_iter)
assert not items_list
pp.pprint(items_list)

14:33:55.065 [INFO] (rs_client.rs_client) Retrieving items from collection 's1_pedc' with query params: {'sortby': '+published', 'limit': 2, 'page': 1}.
14:33:55.200 [INFO] (rs_client.rs_client) Retrieving items from collection 's1_pedc' with query params: {'sortby': '+published', 'limit': 2, 'page': 2}.


[ <Item id=DCS_01_202501270945000000112233>,
  <Item id=DCS_02_202502131123000000987654>]
[]


In [14]:
items = edrs_client.get_items(collection_id='s1_pedc', filter="platform='sentinel-1c'")
items_list = list(items)
assert len(items_list) == 1
pp.pprint(items_list)

14:33:55.351 [INFO] (rs_client.rs_client) Retrieving items from collection 's1_pedc' with query params: {'filter': "platform='sentinel-1c'"}.


[<Item id=DCS_02_202502131123000000987654>]


In [15]:
items = edrs_client.get_items(collection_id='s1_pedc', filter="constellation='sentinel-1'")
items_list = list(items)
assert len(items_list) == 2
pp.pprint(items_list)
#import json
#print(json.dumps(items_list[0].to_dict(), indent=2))

14:33:55.504 [INFO] (rs_client.rs_client) Retrieving items from collection 's1_pedc' with query params: {'filter': "constellation='sentinel-1'"}.


[ <Item id=DCS_02_202502131123000000987654>,
  <Item id=DCS_01_202501270945000000112233>]


In [16]:
# datetime filters
items = edrs_client.get_items(collection_id="s1_pedc", datetime="2026-01-01T00:00:00Z/..")
items_list = list(items)
assert not items_list
items_list

14:33:55.641 [INFO] (rs_client.rs_client) Retrieving items from collection 's1_pedc' with query params: {'datetime': '2026-01-01T00:00:00Z/..'}.


[]

In [17]:
items = edrs_client.get_items(collection_id="s1_pedc", datetime="2024-01-01T00:00:00Z/..")
items_list = list(items)
assert len(items_list) == 2
items_list

14:33:55.780 [INFO] (rs_client.rs_client) Retrieving items from collection 's1_pedc' with query params: {'datetime': '2024-01-01T00:00:00Z/..'}.


[<Item id=DCS_02_202502131123000000987654>,
 <Item id=DCS_01_202501270945000000112233>]

In [18]:
items = edrs_client.get_items(collection_id="s1_pedc", datetime="2024-02-13T11:23:00Z/2025-02-13T11:33:00Z")
items_list = list(items)
assert len(items_list) == 2
items_list

14:33:55.918 [INFO] (rs_client.rs_client) Retrieving items from collection 's1_pedc' with query params: {'datetime': '2024-02-13T11:23:00Z/2025-02-13T11:33:00Z'}.


[<Item id=DCS_02_202502131123000000987654>,
 <Item id=DCS_01_202501270945000000112233>]

In [19]:
items = edrs_client.get_items(collection_id="s1_pedc", datetime="../2025-02-13T12:00:00Z")
items_list = list(items)
assert len(items_list) == 2
items_list

14:33:56.057 [INFO] (rs_client.rs_client) Retrieving items from collection 's1_pedc' with query params: {'datetime': '../2025-02-13T12:00:00Z'}.


[<Item id=DCS_02_202502131123000000987654>,
 <Item id=DCS_01_202501270945000000112233>]

In [20]:
items = edrs_client.get_items(collection_id="s1_pedc", datetime="2024-01-01T00:00:00Z/..")
items_list = list(items)
assert len(items_list) == 2
items_list

14:33:56.225 [INFO] (rs_client.rs_client) Retrieving items from collection 's1_pedc' with query params: {'datetime': '2024-01-01T00:00:00Z/..'}.


[<Item id=DCS_02_202502131123000000987654>,
 <Item id=DCS_01_202501270945000000112233>]

In [21]:
items = edrs_client.get_items(
    collection_id="s1_pedc",
    filter="published='2025-02-13T11:28:42Z'",
)
print(list(items))

items = edrs_client.get_items(
    collection_id="s2_pedc",
    filter="start_datetime='2025-04-19T12:37:00.000Z'",
)
print(list(items))

items = edrs_client.get_items(
    collection_id="s1_pedc",
    filter="end_datetime='2024-04-10T08:42:14Z'",
)
print(list(items))

14:33:56.448 [INFO] (rs_client.rs_client) Retrieving items from collection 's1_pedc' with query params: {'filter': "published='2025-02-13T11:28:42Z'"}.
14:33:56.618 [INFO] (rs_client.rs_client) Retrieving items from collection 's2_pedc' with query params: {'filter': "start_datetime='2025-04-19T12:37:00.000Z'"}.
14:33:56.725 [INFO] (rs_client.rs_client) Retrieving items from collection 's1_pedc' with query params: {'filter': "end_datetime='2024-04-10T08:42:14Z'"}.


[<Item id=DCS_02_202502131123000000987654>]
[<Item id=DCS_03_202504191237000000445566>]
[<Item id=DCS_01_202501270945000000112233>]


In [23]:
items_iter = edrs_client.get_items(
    collection_id='s1_pedc',
    filter="platform='sentinel-1c' AND constellation='sentinel-1'",
    sortby='-published',
    limit=2,
    page=1,
)
assert len(items_list) == 2
items_list

14:34:03.593 [INFO] (rs_client.rs_client) Retrieving items from collection 's1_pedc' with query params: {'filter': "platform='sentinel-1c' AND constellation='sentinel-1'", 'sortby': '-published', 'limit': 2, 'page': 1}.


[<Item id=DCS_02_202502131123000000987654>,
 <Item id=DCS_01_202501270945000000112233>]

In [24]:
# cql2-json filter
import json
items_iter = edrs_client.get_items(
    collection_id="s1_pedc",
    **{
        "filter-lang": "cql2-json",
        "filter": json.dumps({
            "op": "=",
            "args": [
                {"property": "published"},
                {"literal": "2025-02-13T11:28:42Z"},
            ],
        }),
    },
)
items_list = list(items_iter)
assert len(items_list) == 1
items_list

14:34:06.032 [INFO] (rs_client.rs_client) Retrieving items from collection 's1_pedc' with query params: {'filter-lang': 'cql2-json', 'filter': '{"op": "=", "args": [{"property": "published"}, {"literal": "2025-02-13T11:28:42Z"}]}'}.


[<Item id=DCS_02_202502131123000000987654>]

In [25]:
items_iter = edrs_client.get_items(
    collection_id="s1_pedc",
    **{
        "filter-lang": "cql2-json",
        "filter": json.dumps({
            "op": "and",
            "args": [
                {
                    "op": "=",
                    "args": [
                        {"property": "platform"},
                        {"literal": "sentinel-1c"},
                    ],
                },
                {
                    "op": "=",
                    "args": [
                        {"property": "constellation"},
                        {"literal": "sentinel-1"},
                    ],
                },
            ],
        }),
    },
)
items_list = list(items_iter)
assert len(items_list) == 1
items_list

14:34:08.183 [INFO] (rs_client.rs_client) Retrieving items from collection 's1_pedc' with query params: {'filter-lang': 'cql2-json', 'filter': '{"op": "and", "args": [{"op": "=", "args": [{"property": "platform"}, {"literal": "sentinel-1c"}]}, {"op": "=", "args": [{"property": "constellation"}, {"literal": "sentinel-1"}]}]}'}.


[<Item id=DCS_02_202502131123000000987654>]

In [26]:
## STAGING
items_list_s1_pedc = list(edrs_client.get_items(collection_id="s1_pedc"))
items_list_s2_pedc = list(edrs_client.get_items(collection_id="s2_pedc"))
items_to_stage = [items_list_s1_pedc, items_list_s2_pedc]
items_to_stage

14:34:10.253 [INFO] (rs_client.rs_client) Retrieving all items from collection 's1_pedc'.
14:34:10.391 [INFO] (rs_client.rs_client) Retrieving all items from collection 's2_pedc'.


[[<Item id=DCS_02_202502131123000000987654>,
  <Item id=DCS_01_202501270945000000112233>],
 [<Item id=DCS_03_202504191237000000445566>]]

In [27]:
# Create a test collection 
CATALOG_COLLECTION_ID = "SPRINT30_EDRS_TEST_COLLECTION"
collection = create_test_collection(CATALOG_COLLECTION_ID)
items = catalog_client.get_items(CATALOG_COLLECTION_ID)
list(items)

14:34:11.451 [INFO] (rs_client.rs_client) Retrieving all items from collection 'abutu:SPRINT30_EDRS_TEST_COLLECTION'.


[]

In [28]:
init_dask_cluster_staging(scale=2)

# Reload the global vars again
from resources.utils import *  
from resources.dask_utils import *  

pp = pprint.PrettyPrinter(indent=2, width=80, sort_dicts=False, compact=True)

# You can check here the number of workers, threads and memory per worker.
# In local mode, you can configure them by running e.g.
# DASK_MEMORY_EOPF=4G DASK_CORES_EOPF=4 docker compose up # ...
display(dask_cluster_staging)

Connecting to dask gateway for 'dask-staging': http://dask-staging:8000 ...
Create new dask cluster
Dask dashboard for 'dask-staging': http://localhost:8701/clusters/970ba91bfc0544debd723a0038527c09/status


/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| lz4     | 4.4.4  | 4.4.5     | None    |
| msgpack | 1.1.0  | 1.1.1     | None    |
| tornado | 6.3.3  | 6.5.2     | None    |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


Dask workers for 'dask-staging' are up: 0/2
Dask workers for 'dask-staging' are up: 2/2


In [29]:
staging_resp_list = []
for items in items_list:
    staging_resp_list.append(staging_client.run_staging(items.to_dict(), CATALOG_COLLECTION_ID))

for resp in staging_resp_list:
    staging_client.wait_for_jobs(resp, logger)

14:34:28.046 [INFO] (resources.utils) job_status: {'status': 'running', 'message': 'Successfully searched catalog', 'progress': 0, 'type': 'process', 'processID': 'staging', 'created': '2025-11-21T14:34:27Z', 'started': '2025-11-21T14:34:27Z', 'updated': '2025-11-21T14:34:27Z', 'jobID': '93c75b5a-8cd1-4ca6-b5c8-853b173796cf'}
14:34:28.050 [INFO] (resources.utils) ----- Staging from 'pedc' job '93c75b5a-8cd1-4ca6-b5c8-853b173796cf': RUNNING 

14:34:30.088 [INFO] (resources.utils) job_status: {'status': 'running', 'message': 'Successfully searched catalog', 'progress': 0, 'type': 'process', 'processID': 'staging', 'created': '2025-11-21T14:34:27Z', 'started': '2025-11-21T14:34:27Z', 'updated': '2025-11-21T14:34:27Z', 'jobID': '93c75b5a-8cd1-4ca6-b5c8-853b173796cf'}
14:34:30.091 [INFO] (resources.utils) ----- Staging from 'pedc' job '93c75b5a-8cd1-4ca6-b5c8-853b173796cf': RUNNING 

14:34:32.145 [INFO] (resources.utils) job_status: {'status': 'running', 'message': 'Successfully searched ca

In [30]:
# Check that each of the job previously launched are successful
for entry in staging_resp_list:
    inner = next(iter(entry.values()))
    job_id = inner["jobID"]
    job_results = staging_client.get_job_results(job_id)
    print(f"Results from job {job_id}: {job_results}")
    assert job_results == "successful"

Results from job 93c75b5a-8cd1-4ca6-b5c8-853b173796cf: successful


In [31]:
result = list(catalog_client.get_collection(CATALOG_COLLECTION_ID).get_items())
result

[<Item id=DCS_02_202502131123000000987654>]

In [32]:
# DELETE THE WHOLE COLLECTION
result = catalog_client.remove_collection(CATALOG_COLLECTION_ID)
assert result.json()["deleted collection"] == CATALOG_COLLECTION_ID
pp.pprint(result.json())

{'deleted collection': 'SPRINT30_EDRS_TEST_COLLECTION'}


In [ ]:
shutdown = False
if shutdown:    
    # shutdown the clusters
    shutdown_dask_clusters(dask_gateway_staging, dask_cluster_staging.name)
 
# NOTE: restart your python kernel or terminal after the shutdown
# to avoid strange behaviour.